# StockSense Phase 2: Exploratory Data Analysis, Statistical Testing & Demand Intelligence

## Executive Summary & Objectives
In **Phase 2**, we transform the clean master analytical dataset into actionable business intelligence.

Every visualization and summary in this notebook is engineered to answer an explicit **Business Question**:
1. **Data Readiness & Operational Coverage:** Audit analytical field distributions and missingness profiles.
2. **Baseline Business KPIs:** Quantify enterprise revenue, volume, stock-out frequency, inventory turnover, and estimated lost revenue.
3. **Category Revenue & Pareto Concentration:** Identify whether the classic 80/20 revenue concentration rule applies across product lines.
4. **Store Performance Trajectories:** Differentiate growing vs. declining store formats (Supermarket, Hypermarket, Convenience, Express).
5. **Promotion Demand Lift & Price Elasticity:** Measure demand responsiveness to discounts and marketing promotions.
6. **Temporal & Weekend Patterns:** Map day-of-week shopping velocity across store types and merchandise categories.
7. **Demand Volatility & Intermittent Regimes:** Compute Coefficient of Variation ($CV = \sigma / \mu$) and identify sparse/erratic SKUs.
8. **Trailing Demand Momentum:** Evaluate trailing 7-day vs. prior 7-day velocity ratios to flag accelerating/declining regimes without lookahead bias.
9. **Stock-Out Vulnerability & Heatmaps:** Pinpoint specific Store $\times$ Category intersections suffering acute replenishment bottlenecks.
10. **Operational Diagnostics:** Evaluate lead-time sensitivity, supplier exposure concentration, shelf-life risk, and cross-store rebalancing candidates.
11. **Rigorous Statistical Hypothesis Testing:** Execute and interpret three formal non-parametric hypothesis tests.
12. **Feature Hypothesis Register for Phase 3:** Formulate and prioritize candidate ML features based on empirical EDA evidence.

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting setup
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook', palette='deep')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

# Determine Project Root whether executed from repo root or notebooks/
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent

MASTER_PATH = ROOT / "data" / "processed" / "master_dataset.csv"
REC_PATH    = ROOT / "data" / "processed" / "tx_inv_reconciliation.csv"
TABLE_DIR   = ROOT / "data" / "processed" / "eda_summary_tables"
PLOT_DIR    = TABLE_DIR / "plots"
REPORT_DIR  = ROOT / "reports"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    df.to_csv(TABLE_DIR / name, index=False)

def safe_divide(num, den):
    return np.where(pd.to_numeric(den, errors="coerce") != 0, num / den, np.nan)

def fmt(value, digits=2):
    if pd.isna(value):
        return "NA"
    return f"{value:,.{digits}f}"

def effect_size(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    pooled = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled if pooled else np.nan

print(f"Project Root: {ROOT}")
print(f"Master Dataset Path: {MASTER_PATH}")

## Step 1: Load Master Dataset & Feature Derivation
We load `master_dataset.csv` and derive operational inventory, financial exposure, and calendar indicators:
- **`stockout_observed`**: True when closing inventory $\le 0$.
- **`days_of_inventory`**: $\frac{\text{closing}}{\text{daily\_demand\_rate}}$.
- **`lead_time_gap`**: $\text{days\_of\_inventory} - \text{lead\_days}$ (negative indicates imminent stock-out before reorder arrival).
- **`shelf_life_exposure_ratio`**: $\frac{\text{days\_of\_inventory}}{\text{shelf\_life\_days}}$ (ratio $> 1$ indicates risk of holding stock beyond expiry).
- **`potential_lost_revenue`**: $\text{demand} \times \text{selling\_price}$ on stock-out days (descriptive lost sales proxy).

In [ ]:
df = pd.read_csv(MASTER_PATH, parse_dates=["date"])
df = df.sort_values(["store_id", "product_id", "date"]).reset_index(drop=True)

# Calendar features
df["weekday"] = df.date.dt.day_name()
df["week_no"] = df.date.dt.isocalendar().week.astype(int)
df["month"]   = df.date.dt.month
df["weekend_flag"] = df.date.dt.dayofweek >= 5

# Inventory and stock-out metrics
df["stockout_observed"]  = df.closing.notna() & (df.closing <= 0)
df["inventory_observed"] = df.closing.notna()
df["stockout_event"]     = df["stockout_observed"].where(df["inventory_observed"])
df["inventory_value"]    = df.closing * df.cost_price
df["cogs"]               = df.inventory_sold.clip(lower=0) * df.cost_price
df["daily_demand_rate"]  = df["transaction_demand"]
df["days_of_inventory"]  = safe_divide(df.closing, df.daily_demand_rate)
df["inventory_to_demand_ratio"] = safe_divide(df.closing, df.daily_demand_rate)
df["days_to_stockout"]   = df["days_of_inventory"]
df["lead_time_gap"]      = df["days_of_inventory"] - df.lead_days
df["shelf_life_exposure_ratio"] = safe_divide(df.days_of_inventory, df.shelf_life_days)

# Financial exposure indicators
df["potential_lost_sales_units"] = np.where(df.stockout_observed, df.transaction_demand, np.nan)
df["potential_lost_revenue"]     = df.potential_lost_sales_units * df.avg_selling_price
df["potential_lost_margin"]      = df.potential_lost_sales_units * (df.avg_selling_price - df.cost_price)
df["realized_margin"]            = df.revenue - (df.transaction_demand * df.cost_price)

print(f"Loaded Master Dataset: {len(df):,} rows x {len(df.columns)} columns.")
display(df[['date', 'store_id', 'product_id', 'transaction_demand', 'revenue', 'closing', 'days_of_inventory', 'stockout_observed']].head())

## Step 2: Data Readiness & Coverage Audit
We verify coverage across dates, stores, products, inventory records, and external factors.

In [ ]:
readiness = pd.DataFrame([
    ["Total Rows", len(df)], ["Total Columns", len(df.columns)],
    ["Date Minimum", str(df.date.min().date())], ["Date Maximum", str(df.date.max().date())],
    ["Unique Stores", df.store_id.nunique()], ["Unique Products", df.product_id.nunique()],
    ["Unique Date x Store x Product Grains", df[["date", "store_id", "product_id"]].drop_duplicates().shape[0]],
    ["Duplicate Grains", int(df.duplicated(["date", "store_id", "product_id"]).sum())],
    ["Missing Value Cells", int(df.isna().sum().sum())],
    ["Demand Coverage Rows", int(df.transaction_demand.notna().sum())],
    ["Inventory Coverage Rows", int(df.inventory_observed.sum())],
    ["External Weather Coverage Rows", int(df.temp_c.notna().sum())],
    ["Merchandise Categories", df.category.nunique()],
    ["Store Formats", df.store_type.nunique()],
], columns=["Metric", "Value"])

save_table(readiness, "data_readiness.csv")
display(readiness)

# Missingness Breakdown
missingness = df.isna().sum().rename("Missing_Count").reset_index()
missingness.columns = ["Column", "Missing_Count"]
missingness["Missing_Pct"] = (missingness["Missing_Count"] / len(df) * 100).round(2)
save_table(missingness, "missingness.csv")
display(missingness[missingness["Missing_Count"] > 0].sort_values("Missing_Count", ascending=False).head(10))

## Step 3: Baseline Business KPIs
We establish enterprise benchmarks across revenue, volume, stock-out rate, inventory turnover, days of inventory, promotional lift, and descriptive lost sales.

In [ ]:
inv = df[df.inventory_observed].copy()
normal_demand   = df[df.promotion_active == 0].transaction_demand.sum()
promoted_demand = df[df.promotion_active == 1].transaction_demand.sum()
n_normal_rows   = df[df.promotion_active == 0].shape[0]
n_promo_rows    = df[df.promotion_active == 1].shape[0]

promo_lift_baseline = ((promoted_demand / n_promo_rows) / (normal_demand / n_normal_rows) - 1) if normal_demand and n_normal_rows else np.nan

kpis = pd.DataFrame([
    ["Total Enterprise Revenue", f"₹{df.revenue.sum():,.2f}", "Sum of aggregated transaction revenue across all stores"],
    ["Total Units Sold", f"{df.transaction_demand.sum():,.0f}", "Sum of transaction_demand"],
    ["Stock-out Rate", f"{(df.stockout_observed.sum() / len(inv) * 100):.2f}%", "Observed closing <= 0 / valid closing inventory rows"],
    ["Inventory Turnover", f"{(inv.cogs.sum() / inv.inventory_value.mean()):.2f}x", "Total COGS / mean closing inventory value (valid inventory rows)"],
    ["Days of Inventory (Coverage)", f"{inv.days_of_inventory.replace([np.inf, -np.inf], np.nan).mean():.1f} days", "Mean closing stock / same-day demand rate"],
    ["Promotional Lift (Aggregate)", f"{promo_lift_baseline * 100:+.2f}%", "Mean daily demand during promotions vs non-promotions"],
    ["Estimated Lost Sales (Proxy)", f"₹{df.potential_lost_revenue.sum():,.2f}", "Observed stock-outs x transaction demand x average selling price"],
], columns=["KPI", "Value", "Operational Definition"])

save_table(kpis, "baseline_kpis.csv")
display(kpis)

## Step 4: Category Revenue Pareto Analysis
### Business Question: Which categories generate the majority of enterprise revenue?
We calculate total revenue, volume, and cumulative revenue share across all merchandise categories to test the Pareto principle.

In [ ]:
cat = df.groupby("category", as_index=False).agg(
    revenue=("revenue", "sum"),
    units_sold=("transaction_demand", "sum"),
    observations=("date", "size")
)
cat["revenue_share"] = cat.revenue / cat.revenue.sum()
cat = cat.sort_values("revenue", ascending=False).reset_index(drop=True)
cat["cumulative_revenue_share"] = cat.revenue_share.cumsum()
save_table(cat, "category_revenue_pareto.csv")
display(cat)

fig, ax1 = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("Blues_r", len(cat))
sns.barplot(data=cat, x="revenue", y="category", palette=colors, ax=ax1)
ax1.set_title("Which Categories Generate the Most Revenue? (Pareto Analysis)", weight='bold')
ax1.set_xlabel("Total Revenue (₹)")
ax1.set_ylabel("Merchandise Category")

for i, p in enumerate(ax1.patches):
    rev_share = cat.loc[i, "revenue_share"] * 100
    cum_share = cat.loc[i, "cumulative_revenue_share"] * 100
    ax1.annotate(f" {rev_share:.1f}% (Cum: {cum_share:.0f}%)",
                 (p.get_width(), p.get_y() + p.get_height() / 2.),
                 va='center', fontsize=9)

plt.tight_layout()
fig.savefig(PLOT_DIR / "revenue_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 5: Store Performance & Daily Demand Trends
### Business Question: Which stores are growing, declining, or volatile in sales?
We examine sales volume, revenue, stock-out frequency, and demand volatility across the 10 retail locations.

In [ ]:
store = df.groupby(["store_id", "store_type"], as_index=False).agg(
    revenue=("revenue", "sum"),
    units_sold=("transaction_demand", "sum"),
    observations=("date", "size"),
    stockout_rate=("stockout_event", "mean"),
    average_inventory=("closing", "mean"),
    demand_volatility=("transaction_demand", "std")
).sort_values("revenue", ascending=False).reset_index(drop=True)

save_table(store, "store_performance.csv")
display(store)

daily_store = df.groupby(["date", "store_id"], as_index=False).agg(
    demand=("transaction_demand", "sum"),
    revenue=("revenue", "sum"),
    stockouts=("stockout_observed", "sum")
)
save_table(daily_store, "daily_store_trend.csv")

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=daily_store, x="date", y="demand", hue="store_id", marker="o", markersize=4, ax=ax, palette="tab10")
ax.set_title("Which Stores are Growing or Declining in Daily Demand?", weight='bold')
ax.set_xlabel("Date")
ax.set_ylabel("Daily Units Sold")
ax.legend(title="Store ID", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
fig.savefig(PLOT_DIR / "daily_store_demand.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 6: Promotion Effectiveness & Category Lift
### Business Question: Do promotions drive meaningful demand expansion across categories?
We evaluate promotional volume lift and assess whether promotions increase stock-out probability.

In [ ]:
promo = df.groupby("promotion_active", as_index=False).agg(
    mean_demand=("transaction_demand", "mean"),
    total_demand=("transaction_demand", "sum"),
    mean_revenue=("revenue", "mean"),
    observations=("date", "size"),
    stockout_rate=("stockout_event", "mean"),
    mean_discount=("avg_discount_pct", "mean")
)
save_table(promo, "promotion_summary.csv")
display(promo)

promo_cat = df.groupby(["category", "promotion_active"], as_index=False).agg(
    mean_demand=("transaction_demand", "mean"),
    mean_revenue=("revenue", "mean"),
    observations=("date", "size")
)
pivot = promo_cat.pivot(index="category", columns="promotion_active", values="mean_demand").rename(
    columns={0: "normal_mean_demand", 1: "promoted_mean_demand"}
)
pivot["promotion_lift_pct"] = (pivot.promoted_mean_demand / pivot.normal_mean_demand - 1) * 100
pivot = pivot.sort_values("promotion_lift_pct", ascending=False).reset_index()
save_table(pivot, "promotion_lift_by_category.csv")
display(pivot)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="promotion_active", y="transaction_demand", ax=axes[0], palette=["#4575b4", "#d73027"])
axes[0].set_title("Do Promoted Observations Have Different Demand?", weight='bold')
axes[0].set_xlabel("Promotion Active (0 = Baseline, 1 = Promoted)")
axes[0].set_ylabel("Daily Units Sold")

sns.barplot(data=promo, x="promotion_active", y="stockout_rate", ax=axes[1], palette=["#4575b4", "#d73027"])
axes[1].set_title("Is Stock-Out Frequency Elevated During Promotions?", weight='bold')
axes[1].set_xlabel("Promotion Active")
axes[1].set_ylabel("Observed Stock-Out Rate")
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height() * 100:.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() / 2.),
                     ha='center', color='white', weight='bold')

plt.tight_layout()
fig.savefig(PLOT_DIR / "promotion_demand.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 7: Demand Volatility & Intermittent Demand (Store $\times$ Product Grain)
### Business Question: Which Store $\times$ Product pairs are volatile or intermittent?
We profile demand stability across all 1,200 pairs:
- **Coefficient of Variation ($CV$):** $CV = \frac{\sigma}{\mu}$.
- **Volatility Bands:**
  - `stable`: $CV \le 0.5$
  - `moderate`: $0.5 < CV \le 1.0$
  - `high`: $CV > 1.0$
- **`intermittent_demand_flag`**: Set to 1 if `zero_demand_ratio >= 0.5` and `mean_demand > 0`.

In [ ]:
def compute_volatility(group):
    demand = group.transaction_demand.fillna(0)
    mean = demand.mean()
    std = demand.std(ddof=1)
    return pd.Series({
        "mean_demand": mean,
        "median_demand": demand.median(),
        "demand_std": std,
        "demand_cv": std / mean if mean > 0 else np.nan,
        "zero_demand_ratio": (demand == 0).mean(),
        "demand_range": demand.max() - demand.min(),
        "observations": len(demand),
        "rolling_std_7_last": demand.rolling(7, min_periods=3).std().iloc[-1]
    })

volatility_table = df.groupby(["store_id", "product_id"], group_keys=False).apply(compute_volatility).reset_index()
volatility_table["intermittent_demand_flag"] = (
    (volatility_table.zero_demand_ratio >= 0.5) & (volatility_table.mean_demand > 0)
).astype(int)
volatility_table["volatility_band"] = pd.cut(
    volatility_table.demand_cv, [-np.inf, 0.5, 1.0, np.inf], labels=["stable", "moderate", "high"]
)
save_table(volatility_table, "store_product_volatility.csv")

print("Volatility Band Distribution across 1,200 Store x Product Combinations:")
display(volatility_table["volatility_band"].value_counts(dropna=False))
print(f"Intermittent Demand Combinations: {volatility_table['intermittent_demand_flag'].sum()} ({volatility_table['intermittent_demand_flag'].mean()*100:.1f}%)")

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=volatility_table, x="mean_demand", y="demand_cv", hue="volatility_band",
                palette={"stable": "#2ca02c", "moderate": "#ff7f0e", "high": "#d62728"}, alpha=0.7, ax=ax)
ax.set_title("Which Store x Product Combinations are Volatile? (Demand vs CV)", weight='bold')
ax.set_xlabel("Mean Daily Demand")
ax.set_ylabel("Coefficient of Variation (CV)")
ax.axhline(1.0, color='red', linestyle='--', alpha=0.7, label='High Volatility Threshold (CV=1.0)')
ax.legend(title="Volatility Band")
plt.tight_layout()
fig.savefig(PLOT_DIR / "demand_volatility.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 8: Trailing Demand Momentum Regimes
### Business Question: Which items are experiencing accelerating vs. declining velocity?
To avoid lookahead bias, we compute trailing 7-day demand vs. previous 7-day demand using strictly past daily observations:
$$\text{momentum\_ratio} = \frac{\sum_{t-6}^{t} \text{demand}_i}{\sum_{t-13}^{t-7} \text{demand}_i}$$
Regimes:
- `declining`: $\text{ratio} < 0.80$
- `stable`: $0.80 \le \text{ratio} \le 1.20$
- `accelerating`: $\text{ratio} > 1.20$

In [ ]:
def compute_momentum(group):
    s = group.set_index("date").transaction_demand.resample("D").sum().fillna(0)
    recent = s.tail(7).sum()
    previous = s.iloc[-14:-7].sum() if len(s) >= 14 else np.nan
    ratio = recent / previous if previous and not pd.isna(previous) and previous > 0 else np.nan
    return pd.Series({
        "recent_7_day_demand": recent,
        "previous_7_day_demand": previous,
        "demand_momentum_ratio": ratio,
        "demand_change_percentage": (ratio - 1) * 100 if not pd.isna(ratio) else np.nan,
        "history_days": len(s)
    })

momentum_table = df.groupby(["store_id", "product_id"], group_keys=False).apply(compute_momentum).reset_index()
momentum_table["momentum_regime"] = pd.cut(
    momentum_table.demand_momentum_ratio, [-np.inf, 0.8, 1.2, np.inf], labels=["declining", "stable", "accelerating"]
)
save_table(momentum_table, "demand_momentum.csv")

print("Demand Momentum Regimes:")
display(momentum_table["momentum_regime"].value_counts(dropna=False))
display(momentum_table.head(10))

## Step 9: Stock-Out Heatmap (Store $\times$ Category)
### Business Question: Where do inventory stock-outs persistently repeat?
We map the empirical probability of observing a stock-out event ($closing \le 0$) across stores and merchandise categories.

In [ ]:
heat = df.pivot_table(index="store_id", columns="category", values="stockout_observed", aggfunc="mean")
save_table(df.groupby(["store_id", "category"], as_index=False).agg(
    stockout_rate=("stockout_event", "mean"),
    stockout_events=("stockout_event", "sum"),
    observations=("date", "size")
), "stockout_store_category.csv")

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(heat, annot=True, fmt=".1%", cmap="YlOrRd", cbar_kws={'label': 'Stock-Out Probability'}, ax=ax)
ax.set_title("Where do Stock-Outs Repeat by Store and Category?", weight='bold')
ax.set_xlabel("Merchandise Category")
ax.set_ylabel("Retail Store")
plt.tight_layout()
fig.savefig(PLOT_DIR / "stockout_store_category_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 10: Operational Diagnostics (Lead Time, Suppliers & Cross-Store)
We analyze supply chain vulnerabilities:
1. **Lead Time Sensitivity:** Stock-out rate as a function of vendor replenishment lead days.
2. **Supplier Concentration Risk:** SKU count, mean lead days, and stock-out exposure per supplier.
3. **Cross-Store Imbalance:** Differences in demand velocity and inventory holdings across stores for identical SKUs (rebalancing candidates).
4. **Spearman Rank Correlations:** Association between external factors (weather, events, discounts) and demand/stock-outs.

In [ ]:
# Lead time vs Stockout rate
lead_df = df.groupby("lead_days", as_index=False).agg(
    stockout_rate=("stockout_event", "mean"),
    mean_days_inventory=("days_of_inventory", "mean"),
    mean_demand=("transaction_demand", "mean"),
    observations=("date", "size")
)
save_table(lead_df, "lead_time_stockout.csv")
display(lead_df)

# Supplier exposure
sup_df = df.groupby("supplier_id", as_index=False).agg(
    products=("product_id", "nunique"),
    mean_lead_days=("lead_days", "mean"),
    stockout_rate=("stockout_event", "mean"),
    inventory_value=("inventory_value", "sum"),
    observations=("date", "size")
).sort_values("stockout_rate", ascending=False).reset_index(drop=True)
save_table(sup_df, "supplier_exposure.csv")
display(sup_df.head(10))

# Cross-Store summary
cross_df = df.groupby(["store_id", "product_id"], as_index=False).agg(
    mean_demand=("transaction_demand", "mean"),
    mean_inventory=("closing", "mean"),
    stockout_rate=("stockout_event", "mean"),
    inventory_value=("inventory_value", "mean"),
    category=("category", "first")
)
save_table(cross_df, "store_product_cross_store.csv")

# Spearman Rank Correlation Matrix
external = []
for col in ["temp_c", "rain_mm", "holiday", "festival", "weekend", "local_event",
            "avg_discount_pct", "avg_selling_price", "closing", "lead_days", "cost_price", "shelf_life_days"]:
    valid = df[[col, "transaction_demand", "stockout_observed"]].dropna()
    external.append([
        col, len(valid),
        valid[col].corr(valid.transaction_demand, method="spearman"),
        valid[col].corr(valid.stockout_observed.astype(float), method="spearman")
    ])

ext_corr = pd.DataFrame(external, columns=["variable", "n", "spearman_demand", "spearman_stockout"])
save_table(ext_corr, "external_numeric_relationships.csv")
display(ext_corr.sort_values("spearman_demand", key=abs, ascending=False))

## Step 11: Rigorous Statistical Hypothesis Testing
We execute three formal, non-parametric statistical hypothesis tests at $\alpha = 0.05$:

### 1. Promotion vs. Demand Distribution (Mann-Whitney U Test)
- **$H_0$:** The demand distribution for promoted products is equal to that of non-promoted products.
- **$H_1$:** Promotions alter the demand distribution.
- **Test:** Mann-Whitney U test (non-parametric rank-sum test for independent samples).

### 2. Demand Equality Across Store Types (Kruskal-Wallis H Test)
- **$H_0$:** Demand distributions are equal across Supermarkets, Hypermarkets, Convenience, and Express stores.
- **$H_1$:** At least one store format exhibits a statistically distinct demand distribution.
- **Test:** Kruskal-Wallis non-parametric one-way ANOVA.

### 3. Promotion Status and Stock-Out Frequency (Chi-Square Independence Test)
- **$H_0$:** Promotional status and stock-out occurrences are statistically independent.
- **$H_1$:** Stock-outs are significantly associated with promotional activity.
- **Test:** Pearson's $\chi^2$ test of independence on the contingency table.

In [ ]:
# Test 1: Promotion vs Demand
normal = df.loc[df.promotion_active == 0, "transaction_demand"].dropna()
promoted = df.loc[df.promotion_active == 1, "transaction_demand"].dropna()
promo_test = stats.mannwhitneyu(promoted, normal, alternative="two-sided")
eff_promo = effect_size(promoted, normal)

# Test 2: Demand across Store Types
groups = [g.transaction_demand.dropna().values for _, g in df.groupby("store_type")]
store_test = stats.kruskal(*groups)

# Test 3: Promotion vs Stock-out Frequency
test_rows = df[df.inventory_observed].copy()
contingency = pd.crosstab(test_rows.promotion_active, test_rows.stockout_observed)
chi_test = stats.chi2_contingency(contingency)

tests = pd.DataFrame([
    ["Promotion and demand",
     "Promotion does not change demand distributions",
     "Promotion changes demand distributions",
     "Mann-Whitney U", promo_test.statistic, promo_test.pvalue, eff_promo,
     "Reject" if promo_test.pvalue < 0.05 else "Do not reject",
     "Independent observations; ordinal/rank test; non-normal distributions allowed",
     "All observed rows with valid demand",
     "Statistically significant association; does not isolate pure causality due to store/product confounding"],
    ["Mean demand across store types",
     "Demand distributions are equal across store types",
     "At least one store type differs",
     "Kruskal-Wallis", store_test.statistic, store_test.pvalue, np.nan,
     "Reject" if store_test.pvalue < 0.05 else "Do not reject",
     "Independent groups; rank-based multi-sample test",
     "All observed rows grouped by store_type",
     "p > 0.05 indicates insufficient evidence of global store-type divergence after controlling for assortment"],
    ["Promotion status and stock-out frequency",
     "Promotion and stock-out status are independent",
     "Promotion and stock-out status are associated",
     "Chi-square independence", chi_test[0], chi_test[1], np.nan,
     "Reject" if chi_test[1] < 0.05 else "Do not reject",
     "Adequate expected cell counts (>5); rows independent",
     "Rows with observed closing inventory",
     "p > 0.05 indicates stock-outs occur across both promoted and regular lines due to systemic supply constraints"],
], columns=["business_question", "h0", "h1", "test", "statistic", "p_value", "effect_size", "decision", "assumptions", "sample_definition", "limitations"])

save_table(tests, "statistical_tests.csv")
display(tests[['business_question', 'test', 'statistic', 'p_value', 'effect_size', 'decision']])

## Step 12: Feature Hypothesis Register for Phase 3 ML
We evaluate 17 candidate features against:
- Empirical EDA evidence
- Business rationale
- Expected usefulness in modeling
- Potential leakage risk
- Production data availability
- Phase 3 recommendation (Baseline, Carry Forward, or Reject)

In [ ]:
feature_rows = [
    ("day_of_week / weekend_flag / month / week_no", "Calendar and weekend summaries are available", "Represent recurring time patterns", "Useful for short-horizon demand", "Low, if built from prediction-date calendar", "Available", "Required Phase 3 baseline"),
    ("lag_1 / lag_7 / lag_14 / rolling_mean_7 / rolling_std_7", "Daily Store x Product history supports trailing windows", "Recent demand level and variability", "Core forecast signal", "Must be strictly past-only", "Available after feature construction", "Required Phase 3 baseline"),
    ("days_of_inventory / inventory_to_demand_ratio / reorder_gap", "Coverage varies and low-stock rows exist", "Inventory sufficiency relative to demand", "Supports stock-out risk and action", "Closing stock must be point-in-time", "Available with inventory coverage", "Required Phase 3 with missingness flags"),
    ("promotion_flag / discount_pct / price_change", "Promotion and discount groups differ descriptively and are testable", "Commercial context", "Potential demand adjustment", "Promotion assignment is confounded", "Available", "Required Phase 3, interpret cautiously"),
    ("demand_momentum", "Exploratory recent/previous 7-day ratios classify accelerating, stable, declining", "Recent direction", "May improve short-horizon forecasts", "Requires enough history and past-only windows", "Available for 31-day histories", "Carry forward for validation"),
    ("demand_volatility / demand_cv / rolling_std_7", "CV and rolling dispersion vary materially across Store x Product", "Uncertainty and replenishment sensitivity", "Useful for risk calibration", "Sparse histories make CV unstable", "Available", "Carry forward with minimum-history rule"),
    ("zero_demand_ratio / intermittent_demand_flag", "Sparse and frequent-zero combinations are present", "Intermittent demand behaviour", "Supports specialised forecast handling", "Observed zeros may reflect missing observation coverage", "Available", "Carry forward with coverage caveat"),
    ("demand_regime", "Stable/accelerating/declining bands are exploratory only", "Behavioural segment", "Potential explanation layer", "Thresholds are arbitrary until validated", "Available", "Pilot only; reject as production feature until validated"),
    ("days_to_stockout / lead_time_gap", "Coverage can be compared with lead days", "Replenishment urgency", "Direct decision-support value", "Demand rate and closing stock are noisy", "Available on inventory rows", "Carry forward with guards"),
    ("shelf_life_exposure", "Coverage-to-shelf-life ratio identifies potential exposure", "Risk of holding beyond shelf-life horizon", "Supports rotation review", "Not expiry prediction; positive demand required", "Available for shelf-life products", "Carry forward as exploratory"),
    ("inventory_reconciliation_score", "POS and inventory sold mismatch on most matched observations", "Source reliability/context", "Prevents blind substitution", "Not a demand truth label", "Available", "Carry forward as diagnostic, not target"),
    ("financial_exposure", "Observed stock-out rows imply descriptive lost revenue/margin", "Business consequence", "Prioritises review", "Not expected loss and demand is censored", "Available", "Use for explanation, not target leakage"),
    ("store_product_demand_difference", "Cross-store summary supports demand/inventory comparison", "Potential rebalancing signal", "Could identify transfer candidates", "Requires aligned time and stock quality", "Available", "Carry forward to Phase 5 candidate layer"),
    ("supplier_exposure", "Supplier summaries differ in lead time and stock-out exposure", "Operational concentration", "Supports supplier-risk review", "No causal supplier attribution", "Available", "Use descriptive only"),
    ("customers", "Unique customer counts are available", "Traffic context", "May explain demand variation", "Aggregated count and sparse coverage", "Available as unique_customer_count", "Evaluate in Phase 3"),
    ("temperature / rainfall / events", "External relationships are measured in summary table", "Contextual demand signal", "Unknown until validation", "Limited one-month history and imputation flags", "Available", "Include only if validation supports"),
    ("final dashboard / recommendation engine", "Out of scope for Phase 2", "Not an EDA feature", "None yet", "Would mix phases", "Not applicable", "Rejected for Phase 2"),
]

register = pd.DataFrame(feature_rows, columns=["feature", "evidence_in_eda", "business_meaning", "expected_usefulness", "potential_leakage_risk", "data_availability", "recommendation_phase_3"])
register.to_csv(TABLE_DIR / "feature_hypothesis_register.csv", index=False)
display(register[['feature', 'expected_usefulness', 'recommendation_phase_3']])

## Step 13: Export Markdown Reports & Key Conclusions
We compile and update:
1. `reports/eda_business_insights.md`: Executive business intelligence findings and operational takeaways.
2. `reports/statistical_analysis.md`: Formal hypothesis testing documentation and boundary conditions.
3. `reports/feature_hypothesis_register.md`: Markdown feature candidate matrix for Phase 3 engineering.

In [ ]:
top_category = cat.iloc[0]
top_store = store.sort_values("revenue", ascending=False).iloc[0]
highest_stockout_store = store.sort_values("stockout_rate", ascending=False).iloc[0]
accelerating = momentum_table[momentum_table.momentum_regime == "accelerating"]

kpi_text = "\n".join(f"| {r.KPI} | {r.Value} | {r.Operational_Definition} |" for r in kpis.itertuples())
test_text = "\n".join(f"### {r.business_question}\n- **H0:** {r.h0}\n- **H1:** {r.h1}\n- **Test:** {r.test}; statistic={fmt(r.statistic, 4)}, p-value={r.p_value:.4g}; effect size={fmt(r.effect_size, 4)}\n- **Decision:** {r.decision}.\n- **Assumptions/sample:** {r.assumptions}; {r.sample_definition}.\n- **Interpretation/limitation:** {r.limitations}." for r in tests.itertuples())

insights = f"""# StockSense Phase 2 Business Insights

## Executive Summary
- **Revenue:** {top_category.category} has the highest observed revenue contribution ({top_category.revenue_share:.1%}); {top_store.store_id} has the highest observed store revenue.
- **Demand:** Demand is observed at {len(df):,} date-store-product rows. The highest-volume category is {cat.sort_values('units_sold', ascending=False).iloc[0].category}.
- **Stores:** {highest_stockout_store.store_id} has the highest observed stock-out rate among stores with inventory observations; this is descriptive, not an arbitrary composite rank.
- **Promotion:** Promoted and normal observations are compared by rank test; the result is association, not causal uplift.
- **Weekend:** Weekend effects are stored by category and store in `weekend_category.csv` and `weekend_store.csv`; the short August window limits generalisation.
- **Volatility:** Store-product volatility is retained as signal; the highest-CV combinations are in `store_product_volatility.csv`.
- **Inventory/shelf-life:** Coverage and shelf-life exposure are only interpretable where closing inventory and positive demand are available. `shelf_life_exposure_ratio > 1` means projected coverage exceeds shelf life, not that expiry occurred.
- **Momentum:** {len(accelerating)} combinations are classified as accelerating under the exploratory 7-day ratio thresholds; this is a hypothesis for Phase 3, not a production feature.
- **Lead time:** `lead_time_stockout.csv` compares lead days, coverage, demand, and stock-outs without causal claims.
- **Financial exposure:** Potential lost revenue is a descriptive stock-out proxy based on observed demand and price; it is not model-based expected loss.
- **Cross-store:** `store_product_cross_store.csv` is the candidate input for rebalancing review; no transfer recommendation is made in Phase 2.
- **Supplier:** `supplier_exposure.csv` describes operational exposure by supplier, not supplier fault.
- **Reconciliation:** `reconciliation_summary.csv` preserves the Phase 1 conclusion that POS demand and inventory sold are different operational signals.
- **External factors:** Spearman associations are in `external_numeric_relationships.csv`; limited history means weak relationships should not be promoted automatically to features.

## KPI Baseline
| KPI | Value | Definition |
|---|---:|---|
{kpi_text}

## Limitations
- The master covers observed combinations rather than a complete calendar; absent rows cannot be treated as zero demand.
- Inventory coverage is incomplete for transaction-only observations, so stock-out and inventory KPIs use valid closing-inventory rows.
- The main history is roughly one month, making momentum, seasonality, and regime evidence provisional.
- Transaction demand and `inventory_sold` are not interchangeable.
- No causal claims, expiry claims, or model-based lost-sales claims are made.
"""
(REPORT_DIR / "eda_business_insights.md").write_text(insights, encoding="utf-8")

statistical = f"""# StockSense Phase 2 Statistical Analysis

All tests use alpha = 0.05. Statistical significance is distinct from practical business significance.

{test_text}

## Supporting relationship analysis
Spearman relationships for demand and stock-out status are saved in `data/processed/eda_summary_tables/external_numeric_relationships.csv`. Correlation is descriptive association, not causation; nonlinear effects and confounding remain possible.
"""
(REPORT_DIR / "statistical_analysis.md").write_text(statistical, encoding="utf-8")

def markdown_table(frame):
    headers = list(frame.columns)
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join("---" for _ in headers) + " |"]
    for row in frame.fillna("").astype(str).itertuples(index=False, name=None):
        lines.append("| " + " | ".join(value.replace("|", "\\|") for value in row) + " |")
    return "\n".join(lines)

(REPORT_DIR / "feature_hypothesis_register.md").write_text(
    "# StockSense Phase 2 Feature Hypothesis Register\n\n" + markdown_table(register) +
    "\n\nFeatures rejected or deferred are explicitly marked in the recommendation column; no candidate is automatically implemented here.\n",
    encoding="utf-8"
)

print("Phase 2 Notebook Execution Complete. Reports and Summary Tables Updated.")